<a href="https://colab.research.google.com/github/ziheng-jiao/IntroAI_Project/blob/main/jueceshu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 决策树⾃主实践与SKlearn调⽤

## 1 导入实验中的相关包

In [ ]:
"""
pickle包可以将决策树保存下来，⽅便下次直接调⽤
"""
from matplotlib.font_manager import FontProperties
import matplotlib.pyplot as plt
from math import log
import operator
import pickle

## 2 构建简单的分类数据集

In [ ]:
"""
函数说明：创建测试数据集
Parameters:
 None
Returns:
 dataSet - 数据集
 labels - 分类属性
⽤字典存储决策树结构：
    {'是否参加辅导':{0:{'作业完成情况':{0:'fail', 1:'pass'}}, 1:'pass'}}
学习基础：0代表差，1代表中等，2代表好
出勤情况：0代表低，1代表⾼
是否参加辅导：0代表否，1代表是
作业完成情况：0代表差，1代表中等，2代表好
类别（是否通过考试）：fail代表否，pass代表是
"""
def createDataSet():
    dataSet = [[0, 0, 0, 0, 'fail'],
               [0, 1, 0, 1, 'pass'],
               [0, 1, 1, 1, 'pass'],
               [0, 0, 1, 0, 'fail'],
               [0, 1, 0, 2, 'pass'],
               [1, 0, 0, 0, 'fail'],
               [1, 1, 0, 1, 'pass'],
               [1, 1, 1, 2, 'pass'],
               [1, 0, 1, 1, 'pass'],
               [1, 0, 0, 2, 'fail'],
               [2, 0, 1, 2, 'pass'],
               [2, 1, 1, 2, 'pass'],
               [2, 1, 0, 1, 'pass'],
               [2, 0, 0, 1, 'fail'],
               [2, 0, 1, 0, 'fail']]
    labels = ['学习基础', '出勤情况', '是否参加辅导', '作业完成情况'] [cite: 104]
    return dataSet, labels [cite: 105]

## 3 计算香农熵函数

In [ ]:
def calcShannonEnt(dataSet):
    # 返回数据集的行数 [cite: 115]
    numEntires = len(dataSet) [cite: 115]
    # 保存每个标签(Label)出现次数的“字典” [cite: 116]
    labelCounts = {} [cite: 117]
    # 对每组特征向量进行统计 [cite: 118]
    for featVec in dataSet: [cite: 119]
        # 提取标签(Label)信息 [cite: 120]
        currentLabel = featVec[-1] [cite: 121]
        # 如果标签(Label)没有放入统计次数的字典,添加进去 [cite: 122]
        if currentLabel not in labelCounts.keys(): [cite: 123]
            # 创建一个新的键值对,键为currentLabel值为0 [cite: 124]
            labelCounts[currentLabel] = 0 [cite: 124]
        # Label计数 [cite: 128]
        labelCounts[currentLabel] += 1 [cite: 129]
    # 经验熵(香农熵) [cite: 130]
    shannonEnt = 0.0 [cite: 131]
    # 计算香农 [cite: 132]
    for key in labelCounts: [cite: 133]
        # 选择该标签(Label)的概率 [cite: 134]
        prob = float(labelCounts[key]) / numEntires [cite: 135]
        # 利用公式计算 [cite: 136]
        shannonEnt -= prob * log(prob, 2) [cite: 136]
    # 返回经验熵(香农) [cite: 137]
    return shannonEnt [cite: 138]




## 4 必要的⼯具函数

In [ ]:
# 函数说明:按照给定特征划分数据集
# Parameters:
# dataSet - 待划分的数据集
# axis - 划分数据集的特征索引
# value - 需要返回的特征的值
# Returns:
# retDataSet - 划分后的数据集
def splitDataSet(dataSet, axis, value):
    # 创建返回的数据集列表 [cite: 189, 191]
    retDataSet = []
    # 遍历数据集 [cite: 193]
    for featVec in dataSet:
        # 如果符合条件 [cite: 195]
        if featVec[axis] == value:
            # 去掉axis特征 [cite: 196, 197]
            reducedFeatVec = featVec[:axis]
            # 将符合条件的添加到返回的数据集 [cite: 198, 199]
            reducedFeatVec.extend(featVec[axis+1:])
            # 添加到返回结果集 [cite: 200, 201]
            retDataSet.append(reducedFeatVec)
    # 返回划分后的数据集 [cite: 202, 203]
    return retDataSet

# 函数说明:统计classList中出现次数最多的元素(类标签)
# Parameters:
# classList - 类标签列表
# Returns:
# sortedClassCount[0][0] - 出现次数最多的类标签
def majorityCnt(classList):
    # 定义一个字典 [cite: 210, 212]
    classCount = {}
    # 统计classList中每个元素出现的次数 [cite: 213, 217]
    for vote in classList:
        # 如果元素不在字典中 [cite: 218]
        if vote not in classCount.keys():
            # 创建键值对 [cite: 219, 221]
            classCount[vote] = 0
        # 计数 [cite: 222, 223]
        classCount[vote] += 1
    # 根据字典的值降序排序 [cite: 224, 225]
    sortedClassCount = sorted(classCount.items(),
                              key=operator.itemgetter(1),
                              reverse=True) [cite: 226]
    # 返回出现次数最多的元素 [cite: 227, 228]
    return sortedClassCount[0][0]

## 5 决策树相关函数

In [ ]:
def createTree(dataSet, labels, featLabels):
    # 取分类标签(学生是否通过考试:pass or fail)
    classList = [example[-1] for example in dataSet] [cite: 241]
    # 如果当前数据集中所有样本类别完全相同,则直接返回该类别
    if classList.count(classList[0]) == len(classList):
        return classList[0] [cite: 242]
    # 如果所有特征都已经使用完,则返回出现次数最多的类别
    if len(dataSet[0]) == 1:
        return majorityCnt(classList) [cite: 243, 244]
    # 选择当前信息增益最大的特征作为划分特征
    bestFeat = chooseBestFeatureToSplit(dataSet) [cite: 245, 246]
    # 获取最优特征对应的标签名
    bestFeatLabel = labels[bestFeat] [cite: 247, 248]
    featLabels.append(bestFeatLabel) [cite: 249]
    # 根据最优特征生成当前结点
    myTree = {bestFeatLabel: {}} [cite: 250, 251]
    # 删除已经使用过的特征标签
    del (labels[bestFeat]) [cite: 252, 253]
    # 得到该特征在训练集中的所有取值
    featValues = [example[bestFeat] for example in dataSet] [cite: 254, 255]
    # 去掉重复值
    uniqueVals = set(featValues) [cite: 256, 257]
    # 根据最优特征的不同取值递归构建子树
    for value in uniqueVals:
        myTree[bestFeatLabel][value] = createTree(
            splitDataSet(dataSet, bestFeat, value),
            labels,
            featLabels # 注意：此处文档代码略有残缺，逻辑上需传递featLabels
        ) [cite: 260, 261, 262]
    return myTree [cite: 264]

def getNumLeafs(myTree):
    # 初始化叶子
    numLeafs = 0 [cite: 271, 273]
    # 使用list(myTree.keys())[0]或next(iter(myTree))获取结点属性 [cite: 275, 276]
    firstStr = next(iter(myTree)) [cite: 277]
    # 获取下一组字典
    secondDict = myTree[firstStr] [cite: 278, 279]
    for key in secondDict.keys():
        # 测试该结点是否为字典,如果不是字典,代表此节点为叶子结点
        if type(secondDict[key]).__name__ == 'dict':
            numLeafs += getNumLeafs(secondDict[key]) [cite: 281, 283]
        else:
            numLeafs += 1 [cite: 282, 284]
    return numLeafs [cite: 285]

def getTreeDepth(myTree):
    # 初始化决策树深度
    maxDepth = 0 [cite: 292, 294]
    firstStr = next(iter(myTree)) [cite: 296, 297]
    # 获取下一个字典
    secondDict = myTree[firstStr] [cite: 298, 299]
    for key in secondDict.keys():
        # 测试该结点是否为字典
        if type(secondDict[key]).__name__ == 'dict':
            thisDepth = 1 + getTreeDepth(secondDict[key]) [cite: 304, 305]
        else:
            thisDepth = 1 [cite: 306, 307]
        # 更新最深层数
        if thisDepth > maxDepth:
            maxDepth = thisDepth [cite: 309, 310]
    return maxDepth [cite: 312]

def classify(inputTree, featLabels, testVec):
    # 获取决策树结点
    firstStr = next(iter(inputTree)) [cite: 321, 323]
    # 下一个字典
    secondDict = inputTree[firstStr] [cite: 324, 325]
    # 获取特征索引
    featIndex = featLabels.index(firstStr) [cite: 326, 327]
    for key in secondDict.keys():
        # 判断测试数据对应特征是否等于当前分支
        if testVec[featIndex] == key: [cite: 328, 330]
            # 如果还是字典,递归
            if type(secondDict[key]).__name__ == 'dict':
                classLabel = classify(secondDict[key], featLabels, testVec) [cite: 332, 333]
            else:
                # 叶子结点,直接返回结果
                classLabel = secondDict[key] [cite: 334, 336]
    return classLabel [cite: 337]

## 6 利⽤pickle包储存及读取决策树

In [ ]:
# 函数说明:存储决策树 [cite: 339]
# Parameters:
# inputTree - 已经生成的决策树 [cite: 341, 342]
# filename - 决策树的存储文件名 [cite: 343]
def storeTree(inputTree, filename):
    # 以二进制写模式打开文件 [cite: 351]
    with open(filename, 'wb') as fw:
        # 将决策树字典序列化并保存到文件 [cite: 352]
        pickle.dump(inputTree, fw)

# 函数说明:读取决策树 [cite: 353]
# Parameters:
# filename - 决策树的存储文件名 [cite: 355]
# Returns:
# 决策树字典 [cite: 358]
def grabTree(filename):
    # 以二进制读模式打开文件 [cite: 360]
    fr = open(filename, 'rb')
    # 反序列化并返回决策树 [cite: 361]
    return pickle.load(fr)

## 7 主函数

In [ ]:
# 1. 创建数据集和特征标签
dataSet, features = createDataSet() [cite: 363]
featLabels = [] [cite: 364]

# 2. 构建决策树
myTree = createTree(dataSet, features, featLabels) [cite: 365]
# storeTree(myTree, 'classifierStorage.txt') # 可选：存储决策树 [cite: 365]
# myTree = grabTree('classifierStorage.txt') # 可选：读取决策树 [cite: 365]

# 3. 定义测试数据并进行分类
# 测试数据含义：学习基础差(0)、出勤情况高(1)、参加辅导(1)、作业完成情况中等(1)
testvec = [0, 1, 1, 1] [cite: 367, 368]
result = classify(myTree, featLabels, testvec) [cite: 369]

# 4. 输出分类结果
if result == 'pass': [cite: 370]
    print('通过考试') [cite: 371]
if result == 'fail': [cite: 372]
    print('未通过考试') [cite: 373]

# 5. 打印决策树结构并可视化
print(myTree) [cite: 374]
createPlot(myTree) [cite: 375]

# 6. 打印最优特征信息
# print(dataSet) [cite: 376]
# print(calcShannonEnt(dataSet)) [cite: 377]
print("最优特征索引值:" + str(chooseBestFeatureToSplit(dataSet))) [cite: 378]

## 8 额外内容——SKlearn直接调⽤

### 安装Sklearn包指令：

In [ ]:
pip install scikit-learn

### 数据及处理代码：

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn import tree

# 原始数据集 [cite: 401-463]
dataSet = [[0, 0, 0, 0, 'fail'],
           [0, 1, 0, 1, 'pass'],
           [0, 1, 1, 1, 'pass'],
           [0, 0, 1, 0, 'fail'],
           [0, 1, 0, 2, 'pass'],
           [1, 0, 0, 0, 'fail'],
           [1, 1, 0, 1, 'pass'],
           [1, 1, 1, 2, 'pass'],
           [1, 0, 1, 1, 'pass'],
           [1, 0, 0, 2, 'fail'],
           [2, 0, 1, 2, 'pass'],
           [2, 1, 1, 2, 'pass'],
           [2, 1, 0, 1, 'pass'],
           [2, 0, 0, 1, 'fail'],
           [2, 0, 1, 0, 'fail']]

# 提取特征 X (去掉最后一列) 和目标标签 y (最后一列) [cite: 464-469]
X = [row[:-1] for row in dataSet]
y = [row[-1] for row in dataSet]

# 用 LabelEncoder 把 'fail' 和 'pass' 转换为数值 (fail -> 0, pass -> 1) [cite: 470-472]
le = LabelEncoder()
y = le.fit_transform(y)

# 创建决策树分类器并限制最大深度为 4 [cite: 473]
clf = tree.DecisionTreeClassifier(max_depth=4)

# 使用数据构建决策树 [cite: 474]
clf = clf.fit(X, y)

# 预测新数据 [1, 1, 1, 0] 的结果 [cite: 475]
print(clf.predict([[1, 1, 1, 0]]))
# 预测结果为: 1 (代表 pass) [cite: 475]